In [12]:
from google.colab import drive
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

# ==========================================
# 1. CARGA DE DATOS DESDE GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')
ruta = "/content/drive/MyDrive/semestre 6/sist int/dataset.csv"
df = pd.read_csv(ruta, encoding="latin1")

# Limpieza de espacios en los nombres de las columnas
df.columns = df.columns.str.strip()

# Limpieza de espacios en el contenido del dataset
for col in df.columns:
    if col != 'Disease':
        df[col] = df[col].astype(str).str.strip().replace('nan', np.nan)
    else:
        df[col] = df[col].astype(str).str.strip()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# ==========================================
# 2. TRANSFORMACIÓN BINARIA (ONE-HOT ENCODING)
# ==========================================
symptoms_set = set()
for col in df.columns:
    if col != 'Disease':
        symptoms_set.update(df[col].dropna().unique())

symptoms_list = sorted([s for s in symptoms_set if s and s != 'nan'])

X_rows = []
for idx, row in df.iterrows():
    row_syms = set(row.dropna().values) - {row['Disease']}
    binary_vector = [1 if sym in row_syms else 0 for sym in symptoms_list]
    X_rows.append(binary_vector)

X = pd.DataFrame(X_rows, columns=symptoms_list)
y = df['Disease']



In [14]:
# ==========================================
# 3. ENTRENAMIENTO Y CÁLCULO DE ACCURACY
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=35, random_state=42)
clf.fit(X_train, y_train)

# --- CÁLCULO Y MUESTRA DE ACCURACY ---
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))

print(f"Accuracy en Entrenamiento: {train_acc * 100:.2f}%")
print(f"Accuracy en Prueba:        {test_acc * 100:.2f}%")

# Exportar modelo y síntomas para Gradio
joblib.dump(clf, 'dt_model.joblib')
joblib.dump(symptoms_list, 'symptoms_list.joblib')



Accuracy en Entrenamiento: 92.53%
Accuracy en Prueba:        90.24%


['symptoms_list.joblib']

In [15]:
# ==========================================
# 4. ÁRBOL DE DECISIÓN COMPLETO
# ==========================================
plt.figure(figsize=(60, 25), dpi=300)
plot_tree(
    clf,
    feature_names=symptoms_list,
    class_names=sorted(y.unique()),
    filled=True,
    rounded=True,
    fontsize=6
)
plt.title("Estructura Completa del Árbol de Decisión (Todos los Nodos)", fontsize=20, fontweight='bold')
plt.savefig('decision_tree_completo.png', bbox_inches='tight')
plt.close()



In [16]:
# ==========================================
# 5. MATRIZ DE CORRELACIÓN
# ==========================================
importances = pd.Series(clf.feature_importances_, index=symptoms_list)
top_symptoms = importances.sort_values(ascending=False).head(25).index.tolist()

corr_matrix = X[top_symptoms].corr()

plt.figure(figsize=(14, 11), dpi=200)
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Matriz de Correlación (Top 25 Síntomas por Importancia)", fontsize=14, fontweight='bold')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()

print("Proceso finalizado. Se generaron 'decision_tree_completo.png' y 'correlation_matrix.png'.")

Proceso finalizado. Se generaron 'decision_tree_completo.png' y 'correlation_matrix.png'.


In [17]:
# ==========================================\n"
# 6. INFORMACIÓN DEL DATASET Y MODELO\n"
# ==========================================\n"
print(f"Tamaño total de la base de datos: {df.shape[0]} filas x {df.shape[1]} columnas original")
print(f"Tamaño tras One-Hot Encoding:    {X.shape[0]} muestras x {X.shape[1]} síntomas")
print(f"Conjunto de Entrenamiento (Train): {X_train.shape[0]} muestras ({X_train.shape[0]/X.shape[0]*100:.0f}%)")
print(f"Conjunto de Prueba (Test):        {X_test.shape[0]} muestras ({X_test.shape[0]/X.shape[0]*100:.0f}%)")
print(f"Profundidad máxima configurada:  {clf.max_depth}")
print(f"Profundidad real alcanzada:      {clf.get_depth()}")

Tamaño total de la base de datos: 4920 filas x 18 columnas original
Tamaño tras One-Hot Encoding:    4920 muestras x 131 síntomas
Conjunto de Entrenamiento (Train): 3936 muestras (80%)
Conjunto de Prueba (Test):        984 muestras (20%)
Profundidad máxima configurada:  35
Profundidad real alcanzada:      35
